In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 03. Categorization: what are the orders about?

## Two ways to group orders, and why we use both
First, keyword rules. A human writes a list of words for each topic, and the first matching topic wins. This is transparent. Anyone can read the list in scripts/categories.py and disagree with it. That is a feature.

Second, topic modeling. No human list. An algorithm looks at which words tend to appear together in titles, and groups titles into topics. Here we use NMF, non-negative matrix factorization, from scikit-learn. Think of it as sorting a pile of titles into piles where the same words keep showing up together. It finds patterns a human list might miss, but its piles do not come with names. A human still has to read the top words and name each pile.

Neither method reads the full order. Both work from titles here, plus disposition notes where present. A title is a label, not the whole argument. Treat every category as an approximation.

In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Rows: 10879 | Years: 1791 to 2026


In [3]:
import sys
sys.path.insert(0, str(BASE / "scripts"))
from categories import RULES
for cat, kws in RULES:
    print(cat, "->", ", ".join(kws))
print()
counts = DF.groupby("category").size().sort_values(ascending=False)
print(counts.to_string())
fig, ax = plt.subplots(figsize=(10, 6))
top = counts.head(12).iloc[::-1]
ax.barh(top.index, top.values, color="#5b7fa6")
ax.set_title("Orders by keyword category, top 12")
ax.set_xlabel("Orders")
plt.tight_layout()
plt.savefig(CH / "03_keyword_categories.png")
plt.show()

Emergency and Disaster -> emergency, disaster, hurricane, flood, pandemic, covid, coronavirus, earthquake, drought
Civil Rights and Equality -> civil rights, discrimination, equal employment, equal opportunity, desegregat, segregat, affirmative action, women, disability, handicap
Immigration and Citizenship -> immigration, immigrant, alien, citizenship, naturalization, border, visa, deport, refugee, asylum
Defense and Military -> military, army, navy, air force, armed forces, national guard, defense, war , wartime, soldier, naval, fort , arsenal, militia
Veterans -> veteran
Foreign Policy and Trade -> foreign, tariff, sanction, treaty, ambassador, international, import, export, embargo, diplomat
Health and Public Health -> health, disease, medical, drug, public health, hospital, vaccine, opioid, mental health
Environment and Energy -> environment, pollution, energy, climate, conservation, wildlife, water quality, clean air, clean water, carbon, oil , petroleum, natural gas
Land and Pub

In [4]:
print("Category share by era, percent of rows in each era:")
share = pd.crosstab(DF["era"], DF["category"], normalize="index") * 100
print(share.round(1).to_string())
fig, ax = plt.subplots(figsize=(12, 6))
for cat in ["Land and Public Lands", "Government Administration", "Defense and Military", "Economy and Finance", "Foreign Policy and Trade", "Other or General"]:
    if cat in share.columns:
        ax.plot(share.index, share[cat], marker="o", label=cat)
ax.set_title("Category share by era, selected categories")
ax.set_ylabel("Percent of orders in era")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(CH / "03_category_by_era.png")
plt.show()

Category share by era, percent of rows in each era:
category                      Agriculture and Food  Civil Rights and Equality  Defense and Military  Economy and Finance  Education and Science  Emergency and Disaster  Environment and Energy  Foreign Policy and Trade  Government Administration  Health and Public Health  Immigration and Citizenship  Land and Public Lands  Law Enforcement and Justice  Other or General  Transportation and Infrastructure  Veterans
era                                                                                                                                                                                                                                                                                                                                                                                                                           
Antebellum era                                 0.0                        0.0                   3.8                  0

## Unsupervised topic modeling on titles
TF-IDF turns each title into numbers. Words that are rare across titles but common in one title get higher weight. NMF then finds 10 topics, each a set of words that tend to co-occur. The table below is the model output, not a human story. Read the top words first, then judge whether the topic makes sense.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
texts = DF["title_clean"].fillna("").tolist()
vec = TfidfVectorizer(max_features=4000, stop_words="english", min_df=5)
X = vec.fit_transform(texts)
nmf = NMF(n_components=10, random_state=42, max_iter=300)
W = nmf.fit_transform(X)
terms = vec.get_feature_names_out()
for i, comp in enumerate(nmf.components_):
    top = [terms[j] for j in comp.argsort()[-10:][::-1]]
    print(f"Topic {i}: " + ", ".join(top))
DF["nmf_topic"] = W.argmax(axis=1)
print()
print(DF.groupby("nmf_topic").size().to_string())
fig, ax = plt.subplots(figsize=(10, 5))
DF.groupby("nmf_topic").size().plot(kind="bar", ax=ax, color="#e08a3c")
ax.set_title("Orders by dominant NMF topic from titles")
ax.set_ylabel("Orders")
ax.set_xlabel("Topic number")
plt.tight_layout()
plt.savefig(CH / "03_nmf_topics.png")
plt.show()

Topic 0: exemption, compulsory, age, retirement, william, john, charles, george, harry, james
Topic 1: public, withdrawal, lands, resurvey, revocation, pending, classification, withdrawing, land, california
Topic 2: service, civil, rules, amendment, schedule, subdivision, regard, amendments, amending, paragraph
Topic 3: fair, competition, code, industry, manufacturing, trade, approval, textile, paper, products
Topic 4: executive, order, amendment, amending, revocation, withdrawing, amended, relating, 1933, schedule
Topic 5: board, emergency, investigate, dispute, employees, creating, certain, labor, railroad, company
Topic 6: establishing, refuge, migratory, waterfowl, lake, wildlife, president, national, bird, commission
Topic 7: united, states, regulations, foreign, amending, service, selective, prescribing, governing, commission
Topic 8: certain, secretary, national, department, forest, lands, interior, transfer, transferring, war
Topic 9: appointment, authorizing, mrs, regard, posi

## Validation: hand checking 30 orders
A category system that is never checked is just an opinion. Here, 30 orders were hand coded by reading their titles and, where the title alone was ambiguous, by using the known subject of the order. The hand code is my judgment, not a second dataset. The rule code is the automatic keyword result from the dataset. Agreement means the two labels match exactly.

The sample was chosen to cover famous orders, routine orders, and known hard cases, not a random draw. So the agreement rate describes this hard sample, not all 10,880 rows. Random sampling would likely give a higher rate, because most rows are routine and easy. This sample is deliberately tough, and that is why the rate below is modest.

In [6]:
validation = {'9066': ('Defense and Military', 'Defense and Military'), '8802': ('Civil Rights and Equality', 'Defense and Military'), '9981': ('Civil Rights and Equality', 'Government Administration'), '11246': ('Civil Rights and Equality', 'Civil Rights and Equality'), '13228': ('Government Administration', 'Land and Public Lands'), '12866': ('Government Administration', 'Other or General'), '12871': ('Economy and Finance', 'Economy and Finance'), '14042': ('Health and Public Health', 'Emergency and Disaster'), '14110': ('Education and Science', 'Other or General'), '14438': ('Economy and Finance', 'Other or General'), '14389': ('Foreign Policy and Trade', 'Foreign Policy and Trade'), '12890': ('Other or General', 'Other or General'), '13526': ('Defense and Military', 'Other or General'), '13985': ('Civil Rights and Equality', 'Transportation and Infrastructure'), '14063': ('Economy and Finance', 'Economy and Finance'), '12958': ('Defense and Military', 'Other or General'), '13175': ('Land and Public Lands', 'Land and Public Lands'), '12250': ('Civil Rights and Equality', 'Civil Rights and Equality'), '12612': ('Government Administration', 'Other or General'), '11593': ('Environment and Energy', 'Environment and Energy'), '10925': ('Civil Rights and Equality', 'Civil Rights and Equality'), '9835': ('Government Administration', 'Agriculture and Food'), '10450': ('Government Administration', 'Economy and Finance'), '10730': ('Law Enforcement and Justice', 'Law Enforcement and Justice'), '12148': ('Emergency and Disaster', 'Emergency and Disaster'), '12608': ('Government Administration', 'Other or General'), '13045': ('Health and Public Health', 'Health and Public Health'), '13422': ('Government Administration', 'Other or General'), '13672': ('Civil Rights and Equality', 'Civil Rights and Equality'), '14248': ('Government Administration', 'Other or General')}
rows = []
for eo, (hand, rule) in validation.items():
    sub = DF[DF["executive_order_number"].astype(str) == eo + ".0"]
    if len(sub) == 0:
        sub = DF[DF["executive_order_number"].astype(str) == eo]
    title = sub.iloc[0]["title_clean"] if len(sub) else "NOT FOUND"
    rows.append({"eo": eo, "title": title, "hand_code": hand, "rule_code": rule, "agree": hand == rule})
val = pd.DataFrame(rows)
print(val.to_string(index=False))
print()
agree = int(val["agree"].sum())
print(f"Agreement: {agree} of {len(val)} = {100 * agree / len(val):.0f} percent")
print("Rule code was Other or General for", int((val["rule_code"] == "Other or General").sum()), "of the 30, which is the biggest weakness.")
print("Known substring traps in the sample: branch contains ranch (EO 9835), support contains port (EO 13985), homeland contains land (EO 13228).")
print("Honest accuracy note: keyword rules are a blunt first pass. They are useful for broad shares, not for labeling any single order with confidence. For single order claims, read the order itself.")

   eo                                                                                                                                                                                            title                   hand_code                         rule_code  agree
 9066                                                                                                                                     Authorizing the Secretary of War To Prescribe Military Areas        Defense and Military              Defense and Military   True
 8802 Reaffirming Policy of Full Participation in the Defense Program by All Persons, Regardless of Race, Creed, Color, or National Origin, and Directing Certain Action in Furtherance of Said Policy   Civil Rights and Equality              Defense and Military  False
 9981                                                                                            Establishing the President's Committee on Equality of Treatment and Opportunity in the Armed Servic

In [7]:
print("Where the keyword system fails, by type, in the 30 order sample:")
print("- Other or General fallback: the title used words outside the keyword lists, for example artificial intelligence, housing, classified information, federalism.")
print("- Priority order effects: emergency beats health for COVID orders, because emergency is checked first.")
print("- Substring traps: matching part of a word instead of a whole word.")
print("- True ambiguity: an order can be about two topics at once, for example defense and civil rights in EO 8802. A single label forces a choice, and either choice loses something.")

Where the keyword system fails, by type, in the 30 order sample:
- Other or General fallback: the title used words outside the keyword lists, for example artificial intelligence, housing, classified information, federalism.
- Priority order effects: emergency beats health for COVID orders, because emergency is checked first.
- Substring traps: matching part of a word instead of a whole word.
- True ambiguity: an order can be about two topics at once, for example defense and civil rights in EO 8802. A single label forces a choice, and either choice loses something.
